# Foundations of R Software: Week 11

Lectures 43, 46, 47, 48, 49: Data frames, importing and saving data, descriptive statistics and graphics.

**Setup:** In Colab go to `Runtime > Change runtime type` and select **R** before running.

## Lecture 43: Data Frames

### 1. Load the painters data frame

`library(pkg)` loads a package. `head(df)` shows the first rows. The row names are the painters' names.

In [ ]:
library(MASS)
head(painters)

### 2. Summary of a categorical variable

`df$col` extracts a column. `summary()` is generic: for a factor it returns a frequency table.

In [ ]:
summary(painters$School)

### 3. Attach a data frame

`attach(df)` lets columns be used by name without `df$`. `summary()` of a numeric variable gives Min, quartiles, Median, Mean and Max.

In [ ]:
attach(painters)
summary(School)
summary(Composition)

### 4. Detach a data frame

`detach(df)` restores the default, so `painters$` is needed again. `try()` lets the cell continue after the expected error.

In [ ]:
detach(painters)
try(summary(School))

### 5. Subset rows with subset()

`subset(df, condition)` keeps rows where the condition is TRUE. `==` is the logical equal sign.

In [ ]:
subset(painters, School == "F")

### 6. Subset rows with logical indexing

`df[row_condition, ]` selects rows; the empty position after the comma keeps all columns. `df[["col"]]` extracts a column by name.

In [ ]:
painters[painters[["School"]] == "F", ]

### 7. Subset with a numeric condition

Relational operators: `<`, `<=`, `>`, `>=`, `==`, `!=`.

In [ ]:
subset(painters, Composition <= 6)

### 8. Drop columns with select

`select = c(-3, -5)` removes the 3rd and 5th columns (Colour and School).

In [ ]:
subset(painters, School == "F", select = c(-3, -5))

### 9. Split a data frame by a factor

`split(df, f)` partitions the rows by the levels of `f` and returns a list of data frames.

In [ ]:
splitted = split(painters, painters$School)
splitted

### 10. Check that each part is a data frame

`is.data.frame(x)` returns TRUE or FALSE. List elements are accessed with `$`.

In [ ]:
is.data.frame(splitted$A)
names(splitted)

## Lecture 46: Importing Data Files

### 11. Install and load packages

`install.packages("pkg")` downloads a package once; `library(pkg)` loads it each session. `readxl` reads Excel files and `writexl` writes them.

In [ ]:
install.packages(c("readxl", "writexl"))
library(readxl)
library(writexl)

### 12. Working directory

`getwd()` shows the current folder and `setwd("path")` changes it. Files are read from and written to this folder.

In [ ]:
getwd()

### 13. Create the sample Excel file

The lecture uses `spexcel.xlsx` with two sheets. `write_xlsx(list(...), path)` writes each list element as a sheet.

In [ ]:
sheet1 = data.frame(
    "Variable 1" = c(1, 2, 3, 4, 5),
    "Variable 2" = c(10, 20, 30, 40, 50),
    "Variable 3" = c(100, 200, 300, 400, 500),
    check.names = FALSE
)
sheet2 = data.frame(
    "Variable 4" = c(6, 7, 8, 9, 10),
    "Variable 5" = c(110, 120, 130, 140, 150),
    "Variable 6" = c(110, 210, 310, 410, 510),
    check.names = FALSE
)
write_xlsx(list(Sheet1 = sheet1, Sheet2 = sheet2), "spexcel.xlsx")

### 14. Read the first sheet

`read_excel("file.xlsx", sheet = n)` returns a tibble. `sheet` can be a position or a name.

In [ ]:
dataspexcel = read_excel("spexcel.xlsx", sheet = 1)
dataspexcel

### 15. Extract variables and compute the mean

Names containing spaces must be wrapped in backticks after `$`.

In [ ]:
dataspexcel$`Variable 1`
dataspexcel$`Variable 2`
mean(dataspexcel$`Variable 1`)

### 16. Read the second sheet

Same function with `sheet = 2`.

In [ ]:
dataspexcel2 = read_excel("spexcel.xlsx", sheet = 2)
dataspexcel2
dataspexcel2$`Variable 4`
dataspexcel2$`Variable 5`
dataspexcel2$`Variable 6`
mean(dataspexcel2$`Variable 6`)

### 17. Read a sheet by name

Pass the sheet name as a string.

In [ ]:
read_excel("spexcel.xlsx", sheet = "Sheet2")

### 18. Limit the number of rows

`n_max` sets the maximum number of data rows read.

In [ ]:
dataspexcel4 = read_excel("spexcel.xlsx", n_max = 3)
dataspexcel4

### 19. Read a cell range

`range` accepts A1 notation (`"A2:B3"`) or R1C1 row-column notation (`"R2C1:R3C2"`, where R2C3 is the cell in row 2, column 3).

In [ ]:
dataspexcel5 = read_excel("spexcel.xlsx", range = "A2:B3", sheet = 1)
dataspexcel5
dataspexcel6 = read_excel("spexcel.xlsx", range = "A2:B3", sheet = 2)
dataspexcel6
read_excel("spexcel.xlsx", range = "R2C1:R3C2", sheet = 1)

### 20. Read an SPSS file

`read.spss()` from the `foreign` package reads `.sav` files. The cell runs only when `datafile.sav` exists in the working directory.

In [ ]:
library(foreign)
if (file.exists("datafile.sav")) {
    data = read.spss("datafile.sav", to.data.frame = TRUE)
    head(data)
}

### 21. Read an HTML table

`readHTMLTable()` from the `XML` package reads tables from an HTML file. A small HTML file is created first.

In [ ]:
install.packages("XML")
library(XML)
writeLines(
    c(
        "<html><body><table>",
        "<tr><th>Name</th><th>Marks</th></tr>",
        "<tr><td>A</td><td>70</td></tr>",
        "<tr><td>B</td><td>85</td></tr>",
        "</table></body></html>"
    ),
    "table.html"
)
data = readHTMLTable("table.html")
data

### 22. Other formats in the foreign package

Readers: `read.octave()`, `read.systat()`, `read.xport()` (SAS), `read.dta()` (Stata). Below, a Stata file is written and read back.

In [ ]:
write.dta(painters, "painters.dta")
stata_data = read.dta("painters.dta")
head(stata_data)

## Lecture 47: Saving and Writing Data Files

### 23. Write a vector to a file

`write(x, file, ncolumns, append = FALSE, sep = " ")` writes data to a file, 5 columns per line by default for numbers.

In [ ]:
x = c(1:100)
write(x, file = "shalabh")
cat(readLines("shalabh"), sep = "\n")

### 24. Control columns and separator

`ncolumns` sets values per line and `sep` sets the separator; `sep = "\t"` gives tab-delimited output.

In [ ]:
write(x, file = "shalabh_tab", ncolumns = 10, sep = "\t")
cat(readLines("shalabh_tab"), sep = "\n")

### 25. Append to an existing file

With `append = TRUE` the new data is added at the end instead of overwriting.

In [ ]:
write(c(101, 102, 103), file = "shalabh", append = TRUE)
tail(readLines("shalabh"), 3)

### 26. Write a CSV file

`write.csv(x, file, row.names = TRUE, na = "NA", quote = TRUE)` writes a data frame with comma-separated values.

In [ ]:
write.csv(painters, file = "painters.csv")
head(read.csv("painters.csv"))

### 27. Write a table file

`write.table(x, file, sep = " ", quote = TRUE, row.names = TRUE, col.names = TRUE, eol = "\n", na = "NA")` is the general form. `quote = FALSE` removes double quotes.

In [ ]:
write.table(painters, file = "painters.txt", sep = "\t", quote = FALSE)
head(read.table("painters.txt", sep = "\t", header = TRUE))

## Lecture 48: Frequencies and Partition Values

### 28. Absolute and relative frequencies for gender

`table(x)` gives absolute frequencies. Dividing by `length(x)` gives relative frequencies. Codes: 1 = male, 2 = female.

In [ ]:
gender = c(1, 2, 1, 2, 1, 1, 1, 2, 1, 1)
gender
table(gender)
table(gender) / length(gender)

### 29. Absolute and relative frequencies for pizza delivery

Directions: 1 = East, 2 = West, 3 = Central.

In [ ]:
direction = c(
    1, 1, 2, 1, 2, 3, 2, 2, 3, 3, 3, 1, 2, 3, 2, 2, 3, 1, 1, 1,
    3, 3, 3, 2, 2, 2, 2, 1, 2, 2, 1, 1, 1, 3, 2, 2, 1, 2, 2, 3,
    3, 2, 1, 2, 2, 3, 1, 1, 2, 1, 2, 3, 2, 3, 2, 2, 3, 1, 2, 1,
    1, 1, 2, 1, 1, 2, 1, 2, 3, 3, 1, 2, 3, 3, 2, 1, 2, 2, 2, 2,
    2, 3, 2, 2
)
table(direction)
table(direction) / length(direction)

### 30. Quartiles

`quantile(x, probs = seq(0, 1, 0.25))` returns the partition values for the given probabilities. Default is quartiles (0%, 25%, 50%, 75%, 100%).

In [ ]:
marks = c(68, 82, 63, 86, 34, 96, 41, 89, 29, 51, 75, 77, 56, 59, 42)
marks
quantile(marks)
quantile(marks, probs = c(0, 0.25, 0.5, 0.75, 1))

### 31. Custom partition values

Change `probs` for quintiles (5 parts), deciles (10 parts) or percentiles (100 parts).

In [ ]:
quantile(marks, probs = c(0, 0.20, 0.4, 0.6, 0.8, 1))
quantile(marks, probs = seq(0, 1, 0.1))

## Lecture 49: Graphics

### 32. Scatter plot of one variable

`plot(x)` plots the values against their index. Height (cm) of 50 persons.

In [ ]:
height = c(
    166, 125, 130, 142, 147, 159, 159, 147, 165, 156,
    149, 164, 137, 166, 135, 142, 133, 136, 127, 143,
    165, 121, 142, 148, 158, 146, 154, 157, 124, 125,
    158, 159, 164, 143, 154, 152, 141, 164, 131, 152,
    152, 161, 143, 143, 139, 131, 125, 145, 140, 163
)
plot(height)

### 33. Coloured scatter plot

`col` sets the colour of the points.

In [ ]:
plot(height, col = "red")

### 34. Bar plot of raw codes (not recommended)

`barplot(height, ...)` draws one bar per value, so raw codes give one bar per observation.

In [ ]:
barplot(gender)

### 35. Bar plot of absolute frequencies

Bar plots are for categorical variables, so pass the frequency table.

In [ ]:
barplot(table(gender))

### 36. Bar plot of relative frequencies

Divide the table by the number of observations.

In [ ]:
barplot(table(gender) / length(gender))

### 37. Bar plot for direction data

Raw data first, then absolute and relative frequencies.

In [ ]:
barplot(direction)
barplot(table(direction))
barplot(table(direction) / length(direction))

### 38. Adding colours

`col` takes a vector with one colour per bar.

In [ ]:
barplot(
    table(direction),
    col = c("red", "green", "blue")
)

### 39. Adding a title

`main` sets the main title.

In [ ]:
barplot(
    table(direction),
    col = c("red", "green", "blue"),
    main = "Directions of food delivery"
)

### 40. Adding a legend

`legend.text` gives the legend labels.

In [ ]:
barplot(
    table(direction),
    col = c("red", "green", "blue"),
    main = "Directions of food delivery",
    legend.text = c("dir1", "dir2", "dir3")
)

### 41. Adding a subtitle

`sub` adds a subtitle below the x-axis.

In [ ]:
barplot(
    table(direction),
    col = c("red", "green", "blue"),
    main = "Directions of food delivery",
    legend.text = c("dir1", "dir2", "dir3"),
    sub = "Three directions"
)

### 42. Adding axis labels

`xlab` and `ylab` label the x and y axes.

In [ ]:
barplot(
    table(direction),
    col = c("red", "green", "blue"),
    main = "Directions of food delivery",
    legend.text = c("dir1", "dir2", "dir3"),
    sub = "Three directions",
    xlab = "Food Delivery Directions",
    ylab = "Number of Deliveries"
)